# Give your agent the web with Tavily

One agent, built twice:

1. **Without web access.** It only knows what the model learned in training.
2. **With Tavily.** The same agent, plus two tools.

Stack: [LangChain Deep Agents](https://docs.langchain.com/oss/python/deepagents/overview) for the agent, a model served by [Nebius Token Factory](https://tokenfactory.nebius.com/), and [Tavily](https://tavily.com) for the web.

## Setup

Copy `.env.example` to `.env` in the repo root and fill in `NEBIUS_API_KEY` and `TAVILY_API_KEY`.

In [ ]:
import time

from deepagents import create_deep_agent
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain_nebius import ChatNebius

load_dotenv(find_dotenv())

model = ChatNebius(
    model="meta-models/muse-spark-open",
    base_url="https://api.tokenfactory.nebius.com/v1/",
)

SYSTEM_PROMPT = f"""Today is {time.strftime("%Y-%m-%d")}. You are a helpful assistant.
Answer the question directly. Cite a source URL for every fact you look up."""


def ask(agent, question):
    """Run the agent, print the tools it called, and show its answer."""
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", None) or []:
            print("→", call["name"], call["args"])
    display(Markdown(result["messages"][-1].text))

## 1. An agent without web access

Pick a question whose answer changed after the model was trained.

In [ ]:
QUESTION = "What are the three newest models on Nebius Token Factory, and when was each one added?"

agent = create_deep_agent(model=model, system_prompt=SYSTEM_PROMPT)

ask(agent, QUESTION)

## 2. The same agent with Tavily

The only change is the `tools` argument. `TavilySearch` finds sources and `TavilyExtract` reads a page in full.

In [ ]:
from langchain_tavily import TavilyExtract, TavilySearch

web_agent = create_deep_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[TavilySearch(max_results=5), TavilyExtract()],
)

ask(web_agent, QUESTION)

## 3. A use case: competitor brief

The same agent handles a multi-step task. It searches, reads the pages that matter, and writes a short brief. Change the company and run it again.

In [ ]:
ask(
    web_agent,
    """Write a one-page competitor brief on Tavily.
Cover: what it does, pricing, the two closest competitors, and news from the last 30 days.
Read the official pricing page instead of relying on search snippets.""",
)

## Where to go next

- `TavilyCrawl`, `TavilyMap` and `TavilyResearch` are in the same `langchain_tavily` package.
- The repo README lists problem statements you can build from this notebook.
- Tavily docs: https://docs.tavily.com